# Empirical Ablation Study: Data Preprocessing Filters

Comparing 7 pipelines (P1–P7) using different gatekeeping strategies before a **MultinomialNB** classifier.

In [1]:
try:
    import google.colab
    from google.colab import drive

    drive.mount('/content/drive')

    dataset_path = "/content/drive/MyDrive/datasets"
    print("environtment: colab")

except ImportError:
    dataset_path = "./datasets"
    print("environtment: local")

environtment: local


In [2]:
import pandas as pd
import glob
import numpy as np
import ast
import csv
import os
import re
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [3]:
all_files = glob.glob(os.path.join(dataset_path, "*.csv"))

df_raw = pd.concat([
    pd.read_csv(
        f,
        engine="python",
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip"
    )
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df_raw)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df_raw.info()

Total rows: 7736
Dari 1 file: ['FinalFile_new(banget).csv']
<class 'pandas.DataFrame'>
RangeIndex: 7736 entries, 0 to 7735
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                7736 non-null   float64
 1   title             7736 non-null   str    
 2   search_role       7736 non-null   str    
 3   job_level         7736 non-null   str    
 4   company           7736 non-null   str    
 5   location          7736 non-null   str    
 6   salary_avg        1649 non-null   float64
 7   extracted_skills  7736 non-null   str    
 8   skills_count      7736 non-null   int64  
 9   job_description   7730 non-null   str    
 10  job_url           7736 non-null   str    
 11  scraped_at        5728 non-null   str    
 12  source            7736 non-null   str    
 13  salary_min        0 non-null      float64
 14  salary_max        0 non-null      float64
 15  salary_display    7736 non-null   str   

## Base Preprocessing
Drop irrelevant columns, remove duplicates, fill nulls — identical to `main.ipynb`.

In [4]:
DROP_COLS = [
    'salary', 'search_role_raw', 'job_level', 'location_raw',
    'salary_display', 'salary_min', 'salary_max', 'salary_avg',
    'company', 'location', 'job_url', 'search_location', 'scraped_at',
]

df_base = df_raw.copy()
df_base = df_base.drop(columns=DROP_COLS, errors='ignore')
df_base = df_base.drop_duplicates()
df_base.dropna(subset=['extracted_skills', 'search_role'], inplace=True)
df_base['job_description'] = df_base['job_description'].fillna('')
df_base['title'] = df_base['title'].fillna('')
df_base = df_base.reset_index(drop=True)

# ── Role normalisation (same as main.ipynb)
df_base['search_role'] = df_base['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})
ROLES_TO_DROP = [
    'Software Engineer', 'Web Developer', 'IT Support', 'IT',
    'Developer', 'Software', 'System Analyst', 'Progammer', 'Software Developer',
]
df_base = df_base[~df_base['search_role'].isin(ROLES_TO_DROP)].reset_index(drop=True)

print(f"Rows after base preprocessing: {len(df_base)}")
print(df_base['search_role'].value_counts())

Rows after base preprocessing: 7300
search_role
Fullstack Developer    1806
DevOps Engineer        1251
Data Analyst            950
Data Engineer           886
Backend Developer       795
Frontend Developer      572
ML Engineer             548
Data Scientist          492
Name: count, dtype: int64


## Feature Engineering Helpers
Replicating helpers from `main.ipynb`.

In [5]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        if isinstance(result, list) and result:
            return ' '.join(
                s.lower().replace(' ', '_').replace('/', '_')
                for s in result
            )
        return ''
    except Exception:
        return ''

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return ' '.join(text.split()).lower()

def clean_jd(text: str, max_chars: int = 1000) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text[:max_chars]
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    tokens = [t for t in text.lower().split() if len(t) >= 4]
    return ' '.join(tokens)

# Build combined_text (the text representation used for filtering + classification)
df_base['skills_text']    = df_base['extracted_skills'].apply(parse_skills)
df_base['title_text']     = df_base['title'].apply(clean_text)
df_base['jd_text']        = df_base['job_description'].apply(clean_jd)
df_base['combined_text']  = (
    df_base['skills_text'] + ' [SEP] ' +
    df_base['title_text']  + ' [SEP] ' +
    df_base['jd_text']
)

df_base = df_base[df_base['skills_text'].str.strip() != ''].reset_index(drop=True)
df_base = df_base[df_base['combined_text'].str.strip() != ''].reset_index(drop=True)

print(f"Rows after feature engineering: {len(df_base)}")
print('\nSample combined_text:')
print(df_base['combined_text'].iloc[0][:300])

Rows after feature engineering: 4449

Sample combined_text:
angular aws azure ci_cd django express fastapi fiber gcp git java javascript kafka kotlin mongodb mysql postgresql python react redis solid spring_boot typescript vue [SEP] fullstack engineer sde 1 [SEP] about about design implement small medium sized features across frontend backend systems build h


## Reference Standard for Cosine Similarity Filters
One canonical text per role, used as the "anchor" in P2–P6.

In [6]:
# Reference standard: one prototypical sentence per role
REFERENCE_STANDARD = {
    'Frontend Developer'  : 'react vue angular typescript javascript css html ui frontend web',
    'Backend Developer'   : 'java golang python php nodejs api backend rest microservices database',
    'Fullstack Developer' : 'fullstack full stack web software react nodejs postgresql docker',
    'Data Analyst'        : 'data analyst analytics sql tableau bi business intelligence reporting',
    'Data Scientist'      : 'data scientist machine learning ai python tensorflow scikit keras',
    'Data Engineer'       : 'data engineer pipeline spark airflow kafka bigquery cloud etl',
    'DevOps Engineer'     : 'devops sre infrastructure kubernetes docker aws azure linux terraform',
    'ML Engineer'         : 'ml machine learning nlp computer vision pytorch tensorflow llm ai',
}

COSINE_THRESHOLD = 0.05   # rows with similarity < threshold are DROPPED

print("Reference Standard defined for roles:", list(REFERENCE_STANDARD.keys()))
print(f"Cosine threshold: {COSINE_THRESHOLD}")

Reference Standard defined for roles: ['Frontend Developer', 'Backend Developer', 'Fullstack Developer', 'Data Analyst', 'Data Scientist', 'Data Engineer', 'DevOps Engineer', 'ML Engineer']
Cosine threshold: 0.05


## Shared Classifier Helper
All pipelines train/evaluate the **same** MultinomialNB model on whatever rows survive filtering.

In [7]:
def train_and_evaluate(df_filtered: pd.DataFrame,
                        pipeline_name: str,
                        test_size: float = 0.20,
                        random_state: int = 42) -> dict:
    """
    Encode labels, vectorise with TF-IDF (1,2), train MultinomialNB,
    return accuracy and supporting metadata.
    """
    if len(df_filtered) < 10:
        print(f"[{pipeline_name}] Not enough rows ({len(df_filtered)}) to train.")
        return {'pipeline': pipeline_name, 'remaining_rows': len(df_filtered), 'accuracy': None}

    le  = LabelEncoder()
    y   = le.fit_transform(df_filtered['search_role'])
    X   = df_filtered['combined_text'].tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # TF-IDF + Non-negative scaler so MultinomialNB receives non-negative values
    vec = TfidfVectorizer(max_features=1500, ngram_range=(1, 2),
                          min_df=2, sublinear_tf=True)
    X_train_v = vec.fit_transform(X_train)
    X_test_v  = vec.transform(X_test)

    # MultinomialNB requires non-negative; TF-IDF with sublinear_tf is always >= 0
    clf = MultinomialNB()
    clf.fit(X_train_v, y_train)

    y_pred   = clf.predict(X_test_v)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"[{pipeline_name}] Remaining rows: {len(df_filtered):,}  |  Test accuracy: {accuracy:.4f}")
    return {
        'pipeline'       : pipeline_name,
        'remaining_rows' : len(df_filtered),
        'accuracy'       : round(accuracy, 4),
    }

In [8]:
def cosine_filter(df: pd.DataFrame,
                   ngram_range: tuple,
                   threshold: float = COSINE_THRESHOLD) -> pd.DataFrame:
    """
    For each row: vectorise combined_text + its role's reference text
    using TF-IDF with the given ngram_range.
    Drop the row if cosine_similarity < threshold.
    """
    keep_mask = []
    for _, row in df.iterrows():
        ref_text = REFERENCE_STANDARD.get(row['search_role'], row['search_role'])
        corpus   = [row['combined_text'], ref_text]
        vec      = TfidfVectorizer(ngram_range=ngram_range)
        try:
            tfidf_mat = vec.fit_transform(corpus)
            score     = cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:2])[0][0]
        except Exception:
            score = 0.0
        keep_mask.append(score >= threshold)

    return df[keep_mask].reset_index(drop=True)

In [9]:
KEYWORD_MAP_P7 = {
    'frontend'        : ['frontend', 'front-end', 'front end', 'react', 'vue', 'angular', 'ui', 'web'],
    'backend'         : ['backend', 'back-end', 'back end', 'java', 'golang', 'python', 'php', 'node', 'api'],
    'fullstack'       : ['fullstack', 'full-stack', 'full stack', 'software', 'web', 'programmer'],
    'data analyst'    : ['analyst', 'analytics', 'data', 'bi ', 'business intelligence'],
    'data scientist'  : ['scientist', 'data', 'machine learning', 'ai '],
    'data engineer'   : ['engineer', 'data', 'pipeline', 'cloud', 'big data'],
    'devops engineer' : ['devops', 'sre', 'reliability', 'infrastructure', 'cloud', 'aws', 'azure'],
    'ml engineer'     : ['ml ', 'machine learning', 'ai ', 'artificial intelligence', 'nlp', 'vision'],
}

def rule_based_filter(df: pd.DataFrame) -> pd.DataFrame:
    """Keep rows where title contains at least one keyword for the search_role."""
    def is_valid(row):
        title = str(row['title']).lower()
        role  = str(row['search_role']).lower()
        for key, kws in KEYWORD_MAP_P7.items():
            if key in role:
                return any(kw in title for kw in kws)
        role_words = role.replace('_', ' ').split()
        return any(word in title for word in role_words if len(word) > 3)
    mask = df.apply(is_valid, axis=1)
    return df[mask].reset_index(drop=True)

## Run All 7 Pipelines

In [10]:
results = []

# P1 — Baseline: no filtering
print("="*60)
print("Running P1 — Baseline (no filter)")
print("="*60)
results.append(train_and_evaluate(df_base, 'P1 — Baseline (No Filter)'))

# P2–P6 — Cosine Similarity with different N-gram ranges
ngram_configs = [
    ('P2', (1, 1), 'Cosine Sim TF-IDF N-gram (1,1)'),
    ('P3', (1, 2), 'Cosine Sim TF-IDF N-gram (1,2)'),
    ('P4', (1, 3), 'Cosine Sim TF-IDF N-gram (1,3)'),
    ('P5', (1, 4), 'Cosine Sim TF-IDF N-gram (1,4)'),
    ('P6', (1, 5), 'Cosine Sim TF-IDF N-gram (1,5)'),
]

for pid, ngram, label in ngram_configs:
    print(f"\n{'='*60}")
    print(f"Running {pid} — {label}")
    print(f"{'='*60}")
    df_filtered = cosine_filter(df_base, ngram_range=ngram, threshold=COSINE_THRESHOLD)
    print(f"  Rows kept after filter: {len(df_filtered):,} / {len(df_base):,}")
    results.append(train_and_evaluate(df_filtered, f'{pid} — {label}'))

# P7 — Rule-Based Keyword Mapping
print(f"\n{'='*60}")
print("Running P7 — Rule-Based Keyword Mapping")
print("="*60)
df_p7 = rule_based_filter(df_base)
print(f"  Rows kept after filter: {len(df_p7):,} / {len(df_base):,}")
results.append(train_and_evaluate(df_p7, 'P7 — Rule-Based Keyword Mapping'))

Running P1 — Baseline (no filter)
[P1 — Baseline (No Filter)] Remaining rows: 4,449  |  Test accuracy: 0.4528

Running P2 — Cosine Sim TF-IDF N-gram (1,1)
  Rows kept after filter: 1,574 / 4,449
[P2 — Cosine Sim TF-IDF N-gram (1,1)] Remaining rows: 1,574  |  Test accuracy: 0.6635

Running P3 — Cosine Sim TF-IDF N-gram (1,2)
  Rows kept after filter: 889 / 4,449
[P3 — Cosine Sim TF-IDF N-gram (1,2)] Remaining rows: 889  |  Test accuracy: 0.7360

Running P4 — Cosine Sim TF-IDF N-gram (1,3)
  Rows kept after filter: 516 / 4,449
[P4 — Cosine Sim TF-IDF N-gram (1,3)] Remaining rows: 516  |  Test accuracy: 0.7308

Running P5 — Cosine Sim TF-IDF N-gram (1,4)
  Rows kept after filter: 308 / 4,449
[P5 — Cosine Sim TF-IDF N-gram (1,4)] Remaining rows: 308  |  Test accuracy: 0.7903

Running P6 — Cosine Sim TF-IDF N-gram (1,5)
  Rows kept after filter: 197 / 4,449


ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: [5]

## Results Summary

In [ ]:
summary_df = pd.DataFrame(results)
summary_df['rows_dropped']   = len(df_base) - summary_df['remaining_rows']
summary_df['drop_rate (%)']  = ((summary_df['rows_dropped'] / len(df_base)) * 100).round(2)
summary_df['accuracy']       = summary_df['accuracy'].apply(lambda x: f"{x:.4f}" if x is not None else 'N/A')

display_cols = ['pipeline', 'remaining_rows', 'rows_dropped', 'drop_rate (%)', 'accuracy']
print("\n" + "="*80)
print(" ABLATION STUDY — FINAL SUMMARY")
print("="*80)
display(summary_df[display_cols].style
    .set_caption("Empirical Ablation Study: 7 Pipeline Comparison")
    .set_properties(**{'text-align': 'left'})
    .highlight_max(subset=['accuracy'], color='#c6efce')
    .highlight_min(subset=['accuracy'], color='#ffc7ce')
)

In [ ]:
numeric_acc = pd.to_numeric(summary_df['accuracy'], errors='coerce')
best_idx    = numeric_acc.idxmax()
best_row    = summary_df.loc[best_idx]

print(f"\n🏆  Best Pipeline : {best_row['pipeline']}")
print(f"    Remaining Rows : {best_row['remaining_rows']:,}")
print(f"    Accuracy       : {best_row['accuracy']}")